In [169]:
# Written by Sebastian Matiz
import pandas as pd
import requests
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error
from sklearn.metrics import accuracy_score
from bs4 import BeautifulSoup
import numpy as np

cols_to_drop_for_player_stats = [
    'comment',
    'player.firstname',
    'player.lastname',
    'player.id',
    'team.id',
    'team.nickname',
    'team.code',
    'team.name',
    'team.logo',
    'game.id',
    'pos'
]

cols_to_drop_for_game_stats = [
    'league',
    'season',
    'stage',
    'officials',
    'timesTied',
    'leadChanges',
    'nugget',
    'date.start',
    'date.end',
    'date.duration',
    'status.clock',
    'status.halftime',
    'status.short',
    'status.long',
    'periods.current',
    'periods.total',
    'periods.endOfPeriod',
    'arena.name',
    'arena.city',
    'arena.state',
    'arena.country',
    'teams.visitors.id',
    'teams.visitors.name',
    'teams.visitors.nickname',
    'teams.visitors.code',
    'teams.visitors.logo',
    'teams.home.id',
    'teams.home.name',
    'teams.home.nickname',
    'teams.home.code',
    'teams.home.logo',
    'scores.visitors.win',
    'scores.visitors.loss',
    'scores.visitors.series.win',
    'scores.visitors.series.loss',
    'scores.visitors.linescore',
    'scores.home.win',
    'scores.home.loss',
    'scores.home.series.win',
    'scores.home.series.loss',
    'scores.home.linescore'
]

In [170]:
# rapidApi headers
headers = {
    "X-RapidAPI-Key": "REDACTED_RAPIDAPI_KEY",
    "X-RapidAPI-Host": "api-nba-v1.p.rapidapi.com"
}

#########################################################
# team code by id
url = "https://api-nba-v1.p.rapidapi.com/teams"
response = requests.get(url, headers=headers).json()['response']
df = pd.DataFrame(response)
df = df.loc[(df['nbaFranchise'] == True) & (df['allStar'] == False)]

team_nickname_to_id_map = {}
for index, row in df.iterrows():
    team_nickname_to_id_map.update({row["nickname"]: row["id"]})
#########################################################

In [171]:
#########################################################
# games by game ids #####################################
def get_games_by_game_ids(season, team):
    url = "https://api-nba-v1.p.rapidapi.com/games"
    querystring = {"season":season,"team":team}

    response = requests.get(url, headers=headers, params=querystring)
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    df = df.loc[(df["status.long"] == "Finished")]
    df = df.sort_values(by=["date.start"])
    
    # adding win col to df
    df['win'] = ''

    df.loc[
        ((df["scores.home.points"] > df["scores.visitors.points"]) & 
        (int(team) == df["teams.home.id"])) |
        ((df["scores.home.points"] < df["scores.visitors.points"]) & 
        (int(team) == df["teams.visitors.id"])),
        'win'
    ] = 1
    
    df.loc[
        ((df["scores.home.points"] < df["scores.visitors.points"]) & 
        (int(team) == df["teams.home.id"])) |
        ((df["scores.home.points"] > df["scores.visitors.points"]) & 
        (int(team) == df["teams.visitors.id"])),
        'win'
    ] = 0
    
    # adding home col to df
    df['home'] = ''
    
    df.loc[(int(team) == df["teams.home.id"]), 'home'] = 1
        
    df.loc[(int(team) == df["teams.visitors.id"]), 'home'] = 0    
    
    return drop_cols(df, cols_to_drop_for_game_stats)
#########################################################

In [172]:
#########################################################
# drop all cols from a df ##############################
def drop_cols(df, cols):
    for col in cols:
        df = df.drop(col, axis=1)
    return df
#########################################################

In [173]:
#########################################################
# get top n player stats by game by #####################
def get_top_players_per_game_df(n, team, game_id):    
    url = "https://api-nba-v1.p.rapidapi.com/players/statistics"

    querystring = {"game": game_id }
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )

    df_opponent = df.loc[df['team.id'] != team]
    df = df.loc[df['team.id'] == team]
    
    df_opponent = drop_cols(df_opponent, cols_to_drop_for_player_stats)
    df = drop_cols(df, cols_to_drop_for_player_stats)
    
    df_opponent = zero_non_numeric_values(df_opponent)
    df = zero_non_numeric_values(df)
    
    df_opponent = df_opponent.add_prefix("opponent.")

    return df_opponent.nlargest(n, "opponent.plusMinus"), df.nlargest(n, "plusMinus")
#########################################################

In [174]:
#########################################################
# flatten data frames ###################################
def flatten_df(df):
    # Flatten the DataFrame
    flattened_data = {}
    for col in df.columns:
        for row in range(df.shape[0]):
            new_col_name = f"{col}{row}"
            flattened_data[new_col_name] = df[col].iloc[row]

    # Convert to DataFrame
    return pd.DataFrame([flattened_data])
#########################################################

In [175]:
#########################################################
# per game, get top 5 players by plusMinus metric #######
def get_top_five_players_per_game(team_id, game_ids): 
    top_5_players_on_team_per_game = {}
    for game_id in game_ids:
        # transform player_stats_df
        opponent_top_five_player_stats, friendly_top_five_player_stats = get_top_players_per_game_df(5, team_id, game_id)
        opponent_top_five_player_stats = flatten_df(opponent_top_five_player_stats)
        friendly_top_five_player_stats = flatten_df(friendly_top_five_player_stats)
        top_five_players_stats = pd.concat(
            [opponent_top_five_player_stats, friendly_top_five_player_stats], 
            axis=1
        )
        ###########################
        top_5_players_on_team_per_game[game_id] = top_five_players_stats          
    return top_5_players_on_team_per_game
#########################################################

In [176]:
#########################################################
# get win prc, and last 10 win prc ######################
def get_win_prc(game_df):
    total_win_prc = []
    last_ten_win_prc = []
    last_ten_games_win_loss = []
    total_games_played = []
    last_ten_win_count = 0
    total_win_count = 0
    total_game_count = 0
    
    for index, row in game_df.iterrows():
        total_game_count += 1
        last_ten_games_win_loss.append(row['win'])
        
        if row['win']:
            total_win_count += 1
            last_ten_win_count += 1
            
        total_win_prc.append(total_win_count/total_game_count)
               
        if total_game_count >= 10:
            if last_ten_games_win_loss[0]:
                last_ten_win_count -= 1
            last_ten_games_win_loss.pop(0)
            last_ten_win_prc.append(last_ten_win_count/10)
        else:
            last_ten_win_prc.append(last_ten_win_count/total_game_count)
        
        total_games_played.append(total_game_count)            
    return last_ten_win_prc, total_win_prc, total_games_played
#########################################################

In [177]:
#########################################################
# combine player stats and games features ###############
import numpy as np
def combine_player_stats_and_games_data(games_df, player_stats_per_game):
    feature_map = []
    feature_map_cols_header = []
    for index, row in games_df.iterrows():
        game_id = row["id"]
        game_df = pd.DataFrame(row).transpose()
        players_stats_df = player_stats_per_game.get(game_id)
        if len(feature_map_cols_header) < 1:
            feature_map_cols_header = list(game_df) + list(players_stats_df)

        game_data = np.array(game_df.iloc[0])
        player_stats_data = np.array(players_stats_df.iloc[0])
        feature_map_data_row = np.concatenate((game_data, player_stats_data))
        feature_map.append(feature_map_data_row)
    return pd.DataFrame(feature_map, columns=feature_map_cols_header)
#########################################################

In [178]:
#########################################################
def get_feature_map_and_y(season, team_id):
    games_df = get_games_by_game_ids(season, team_id) 
    
    players_stats_per_games = get_top_five_players_per_game(
        team_id, 
        games_df["id"].array
    )
    
    last_ten_win_prc, total_win_prc, total_games_played = get_win_prc(games_df)
    games_df = games_df.assign(last_ten_w_prc=last_ten_win_prc)
    games_df = games_df.assign(w_prc=total_win_prc)
    games_df = games_df.assign(games_played=total_games_played)
    
    df = combine_player_stats_and_games_data(games_df, players_stats_per_games)
    df = zero_non_numeric_values(df)
    x = drop_cols(df, ["win", "id"])
    y = df["win"]
    return x, y 
#########################################################

In [179]:
#########################################################
def get_rf_regressor_and_classifier_model(x, y):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
    # Regressor #############################################
    rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

    # Train the model on the training set
    rf_regressor.fit(x_train, y_train)

    # Make predictions
    rf_regressor_predictions = rf_regressor.predict(x_test)
    
    # Evaluate the model
    rf_regressor_mse = mean_squared_error(y_test, rf_regressor_predictions)
    print(f'FR Regressor Mean Squared Error: {rf_regressor_mse}')
    #########################################################

    # Classifier ############################################
    # training random forest classifier 
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

    # Train the model on the training set
    rf_classifier.fit(x_train, y_train)
    
    # Make predictions on the test set
    rf_classifier_predictions = rf_classifier.predict(x_test)

    # Evaluate the model
    rf_classifier_accuracy = accuracy_score(y_test, rf_classifier_predictions)
    print(f'Accuracy: {rf_classifier_accuracy:.2f}')
    #########################################################
    
    return rf_regressor, rf_classifier
#########################################################

In [180]:
#########################################################
def zero_non_numeric_values(df):
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df
#########################################################

In [181]:
#########################################################
rf_model_and_data_map = {}
def get_data_and_rf_models(season, team_nickname):
    team_id = team_nickname_to_id_map.get(team_nickname)
    x, y = get_feature_map_and_y(season, team_id)
    regressor, classifier = get_rf_regressor_and_classifier_model(x, y)
    rf_model_and_data_map.update({team_nickname: [regressor, classifier, x, y]})
#########################################################

In [182]:
#########################################################
def get_player_season_stats_avgs(season, player_id):
    url = "https://api-nba-v1.p.rapidapi.com/players/statistics"
    querystring = {"id":player_id,"season":season}
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    
    df = drop_cols(df, cols_to_drop_for_player_stats)
    df = zero_non_numeric_values(df) 
    
    return df.mean()
#########################################################

In [183]:
#########################################################
def get_players_season_stats_avgs(season, player_ids):
    player_stats_avgs = []
    for i in player_ids:
        player_stats_avgs.append(get_player_season_stats_avgs(season, i))
    
    df = pd.DataFrame(player_stats_avgs)
    return df.sort_values(by=["plusMinus"], ascending=False)
#########################################################

In [184]:
#########################################################
def get_player_lineups_for_tn():
    url = 'https://www.rotowire.com/basketball/nba-lineups.php'
    response = requests.get(url, verify=False)
    soup = BeautifulSoup(response.text, "html.parser")
    players_by_team = {}
    button_divs = soup.find_all("button", class_="see-court-on-off")

    for div in button_divs:
        nickname = div["data-nickname"]
        players_by_team.update({ nickname: [] })
        player_ids = div["data-lineup"].split(",")[:5]
        for player_id in player_ids:
            player_divs = soup.find_all("li", class_="lineup__player is-pct-play-100")
            for player_div in player_divs:
                a_tags = player_div.find_all('a', href=lambda href: href and player_id in href)
                for a_tag in a_tags:
                    players_by_team[nickname].append(a_tag["title"])
            
    return players_by_team
#########################################################

In [185]:
#########################################################
def get_players_by_team_and_season(team_id, season):
    url = "https://api-nba-v1.p.rapidapi.com/players"
    querystring = {"team": team_id,"season":season}
    return pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
#########################################################

In [186]:
#########################################################
def get_team_latest_stats(team_id, season):
    url = "https://api-nba-v1.p.rapidapi.com/teams/statistics"
    querystring = {"id": team_id,"season": season}
    return pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
#########################################################

In [187]:
#########################################################
def get_prediction_feature_map(
    training_data_tail,
    season,
    team_nickname,
    o_team_nickname,
    home
):   
    team_id = team_nickname_to_id_map.get(team_nickname)
    o_team_id = team_nickname_to_id_map.get(o_team_nickname)
    
    # getting player attributes
    df = get_players_by_team_and_season(team_id, season)    
    o_df = get_players_by_team_and_season(o_team_id, season)

    concat_names = df['firstname'] + " " + df['lastname']
    o_concat_names = o_df['firstname'] + " " + o_df['lastname']
    
    name_list = players_name_map.get(team_nickname)
    o_name_list = players_name_map.get(o_team_nickname)
    
    # Filter the DataFrame based on whether the concatenated names exist in name sets
    df_players = df[concat_names.isin(set(name_list))] 
    o_df_players = o_df[o_concat_names.isin(set(o_name_list))]

    team_latest_stats = get_team_latest_stats(team_id, season)
    o_team_latest_stats = get_team_latest_stats(o_team_id, season)

    home_avg_ppg = 0
    visitors_avg_ppg = 0
    if home == 1:
        home_avg_ppg = team_latest_stats.iloc[0]["points"] / team_latest_stats.iloc[0]["games"]
        visitors_avg_ppg = o_team_latest_stats.iloc[0]["points"] / o_team_latest_stats.iloc[0]["games"]
    else:
        home_avg_ppg = o_team_latest_stats.iloc[0]["points"] / o_team_latest_stats.iloc[0]["games"]
        visitors_avg_ppg = team_latest_stats.iloc[0]["points"] / team_latest_stats.iloc[0]["games"]    
    
    player_ids = np.array(df_players["id"])
    o_player_ids = np.array(o_df_players["id"])
    
    players_stats_avgs = get_players_season_stats_avgs(season, player_ids)
    o_players_stats_avgs = get_players_season_stats_avgs(season, o_player_ids)
    o_players_stats_avgs = o_players_stats_avgs.add_prefix("opponent.")

    players_stats_avgs = flatten_df(players_stats_avgs)
    o_players_stats_avgs = flatten_df(o_players_stats_avgs)

    players_stats = pd.concat(
        [o_players_stats_avgs, players_stats_avgs], 
        axis=1
    )
    
    games_col_headers = [
        "scores.visitors.points",
        "scores.home.points", 
        "home", 
        "last_ten_w_prc", 
        "w_prc", 
        "games_played"
    ]
    
    games_data = [
        visitors_avg_ppg, 
        home_avg_ppg, 
        home, 
        training_data_tail["last_ten_w_prc"].iloc[0], 
        training_data_tail["w_prc"].iloc[0],
        training_data_tail["games_played"].iloc[0] + 1
    ]
        
    feature_map_col_headers = games_col_headers + list(players_stats)
    feature_map_data_row = np.concatenate((games_data, players_stats.iloc[0]))
    return pd.DataFrame([feature_map_data_row], columns=feature_map_col_headers)    
#########################################################

In [188]:
players_name_map = get_player_lineups_for_tn()

/home/sebdb/projects/SBS_V1/lib/python3.11/site-packages/urllib3/connectionpool.py:1103: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.rotowire.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [189]:
# Get models for teams ##################################
rf_model_map = {}
print("76ers")
get_data_and_rf_models("2023", "76ers")
print()

print("Knicks")
get_data_and_rf_models("2023", "Knicks")
print()

# print("Pacers")
# pacers_x, pacers_y, pacers_regressor, pacers_classifier = get_data_and_rf_models("2023", "Pacers")
# print()

print("Thunder")
get_data_and_rf_models("2023", "Thunder")
print()

print("Wizards")
get_data_and_rf_models("2023", "Wizards")
print()

# print("Grizzlies")
# grizz_x, grizz_y, grizz_regressor, grizz_classifier = get_data_and_rf_models("2023", "Grizzlies")
# print()

print("Rockets")
get_data_and_rf_models("2023", "Rockets")
print()

# print("Spurs")
# spurs_x, spurs_y, spurs_regressor, spurs_classifier = get_data_and_rf_models("2023", "Spurs")
# print()

print("Celtics")
get_data_and_rf_models("2023", "Celtics")
print()

# print("Jazz")
# jazz_x, jazz_y, jazz_regressor, jazz_classifier = get_data_and_rf_models("2023", "Jazz)
# print()

# print("Timberwolves")
# wolves_x, wolves_y, wolves_regressor, wolves_classifier = get_data_and_rf_models("2023", "Timberwolves")
# print()

print("Clippers")
get_data_and_rf_models("2023", "Clippers")
print()

print("Bucks")
get_data_and_rf_models("2023", "Bucks")
print()

# print("Kings")
# kings_x, kings_y, kings_regressor, kings_classifier = get_data_and_rf_models("2023", "Kings")
# print()

# print("Raptors")
# rapt_x, rapt_y, rapt_regressor, rapt_classifier = get_data_and_rf_models("2023", "Raptors")
# print()
#########################################################

76ers
FR Regressor Mean Squared Error: 0.03481428571428571
Accuracy: 0.93

Knicks
FR Regressor Mean Squared Error: 0.025842857142857143
Accuracy: 1.00

Thunder
FR Regressor Mean Squared Error: 0.08934285714285717
Accuracy: 0.93

Wizards
FR Regressor Mean Squared Error: 0.021328571428571426
Accuracy: 0.93

Rockets
FR Regressor Mean Squared Error: 0.1385357142857143
Accuracy: 0.71

Celtics
FR Regressor Mean Squared Error: 0.06354285714285715
Accuracy: 0.93

Clippers
FR Regressor Mean Squared Error: 0.13672857142857145
Accuracy: 0.93

Bucks
FR Regressor Mean Squared Error: 0.06354
Accuracy: 0.93



In [190]:
# Get models for teams ##################################
# print("Pistons")
# pistons_x, pistons_y, pistons_regressor, pistons_classifier = get_data_and_rf_models("2023", "Pistons")
# print()

# print("Nets")
# nets_x, nets_y, nets_regressor, nets_classifier = get_data_and_rf_models("2023", "Nets")
# print()

# print("Magic")
# magic_x, magic_y, magic_regressor, magic_classifier = get_data_and_rf_models("2023", "Magic")
# print()

# print("Nuggets")
# nugs_x, nugs_y, nugs_regressor, nugs_classifier = get_data_and_rf_models("2023", "Nuggets")
# print()

# print("Heat")
# heat_x, heat_y, heat_regressor, heat_classifier = get_data_and_rf_models("2023", "Heat")
# print()

print("Bulls")
get_data_and_rf_models("2023", "Bulls")
print()

# print("Hornets")
# hornets_x, hornets_y, hornets_regressor, hornets_classifier = get_data_and_models("2023", "Hornets")
# print()

# print("Cavaliers")
# cavs_x, cavs_y, cavs_regressor, cavs_classifier = get_data_and_rf_models("2023", "Cavaliers")
# print()

# print("Pelicans")
# pels_x, pels_y, pels_regressor, pels_classifier = get_data_and_rf_models("2023", "Pelicans")
# print()

# print("Warriors")
# war_x, war_y, war_regressor, war_classifier = get_data_and_rf_models("2023", "Warriors")
# print()

print("Suns")
get_data_and_rf_models("2023", "Suns")
print()

print("Mavericks")
get_data_and_rf_models("2023", "Mavericks")
print()

# print("Lakers")
# lal_x, lal_y, lal_regressor, lal_classifier = get_data_and_rf_models("2023", "Lakers)
# print()

# print("Hawks")
# atl_x, atl_y, atl_regressor, atl_classifier = get_data_and_rf_models("2023", "Hawks)
# print()

print("Trail Blazers")
get_data_and_rf_models("2023", "Trail Blazers")
print()
#########################################################

Bulls
FR Regressor Mean Squared Error: 0.11203333333333333
Accuracy: 0.80

Suns
FR Regressor Mean Squared Error: 0.048042857142857144
Accuracy: 0.93

Mavericks
FR Regressor Mean Squared Error: 0.03503571428571429
Accuracy: 1.00

Trail Blazers
FR Regressor Mean Squared Error: 0.08643571428571427
Accuracy: 0.86



In [197]:
def predict(
    season,
    nickname_0, 
    nickname_1
):
    list(rf_model_and_data_map.get(nickname_0)[2].tail(1))
    
    x_0 = get_prediction_feature_map(rf_model_and_data_map.get(nickname_0)[2].tail(1), season, nickname_0, nickname_1, 0)
    list(x_0)
    x_1 = get_prediction_feature_map(rf_model_and_data_map.get(nickname_1)[2].tail(1), season, nickname_1, nickname_0, 1)
    print(nickname_0, " Regressor: ", rf_model_and_data_map.get(nickname_0)[0].predict(x_0))
    print(nickname_1, " Regressor: ", rf_model_and_data_map.get(nickname_1)[0].predict(x_1))
    print(nickname_0, " Classifier: ", rf_model_and_data_map.get(nickname_0)[1].predict(x_0))
    print(nickname_1, " Classifier: ", rf_model_and_data_map.get(nickname_1)[1].predict(x_1))
    

In [198]:
season = "2023"
predict(season, "Suns", "Celtics")
# predict(season, "Wizards", "Rockets")
# predict(season, "Clippers", "Bulls")
# predict(season, "76ers", "Bucks")
# predict(season, "Mavericks", "Thunder")
# predict(season, "Knicks", "Trail Blazers")

,scores.visitors.points,scores.home.points,home,last_ten_w_prc,w_prc,games_played,opponent.points0,opponent.points1,opponent.points2,opponent.points3,...,blocks0,blocks1,blocks2,blocks3,blocks4,plusMinus0,plusMinus1,plusMinus2,plusMinus3,plusMinus4
69,117.0,111.0,0,0.5,0.6,70,3.0,3.0,17.0,13.0,...,0.0,1.0,0.0,2.0,1.0,13.0,12.0,6.0,5.0,5.0


,scores.visitors.points,scores.home.points,home,last_ten_w_prc,w_prc,games_played,opponent.points0,opponent.points1,opponent.points2,opponent.points3,...,blocks0,blocks1,blocks2,blocks3,blocks4,plusMinus0,plusMinus1,plusMinus2,plusMinus3,plusMinus4
0,117.185714,120.485714,0.0,0.5,0.6,71.0,14.846154,26.6,12.714286,7.842105,...,0.388889,1.0,1.177419,0.650794,0.263158,5.685185,5.538462,4.5,3.31746,2.236842


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- opponent.assists4
- opponent.blocks4
- opponent.defReb4
- opponent.fga4
- opponent.fgm4
- ...
